# COVID-19 Analytics — Exploratory Data Analysis

**Data:** [Our World in Data](https://github.com/owid/covid-19-data) · Jan 2020 – Aug 2024
**Database:** `data/covid_analytics.db` (built by `src/ingest.py`)

This notebook explores the SQLite database with SQL first, then drills down
with pandas/plotly for the visual layer.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd
import plotly.express as px

DB = Path("..") / "data" / "covid_analytics.db"
conn = sqlite3.connect(DB)

px.defaults.template = "plotly_white"
pd.options.display.float_format = "{:,.1f}".format

## 1. Database tour

In [2]:
for t in ("countries", "daily_stats"):
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:12s} {n:>9,} rows")

pd.read_sql_query("SELECT * FROM ingestion_log", conn)

countries          244 rows
daily_stats    401,502 rows


,id,source,ingested_at,rows_loaded,countries_loaded,min_date,max_date
0,1,https://raw.githubusercontent.com/owid/covid-1...,2026-09-10T08:19:38+00:00,401502,244,2020-01-01,2024-08-14


In [3]:
pd.read_sql_query("""
    SELECT * FROM countries LIMIT 5
""", conn)

,country_code,location,continent,population,population_density,median_age,aged_65_older,gdp_per_capita,extreme_poverty,cardiovasc_death_rate,diabetes_prevalence,life_expectancy,human_development_index,hospital_beds_per_thousand
0,FLK,Falkland Islands,South America,"3,801.0",NaN,None,None,None,None,None,NaN,81.4,None,None
1,AIA,Anguilla,North America,"15,877.0",NaN,None,None,None,None,None,NaN,81.9,None,None
2,COK,Cook Islands,Oceania,"17,032.0",NaN,None,None,None,None,None,NaN,76.2,None,None
3,BES,Bonaire Sint Eustatius and Saba,North America,"27,052.0",NaN,None,None,None,None,None,NaN,77.8,None,None
4,VGB,British Virgin Islands,North America,"31,332.0",208.0,None,None,None,None,None,13.7,79.1,None,None


## 2. Pandemic waves

Monthly global cases — each peak is a variant wave (Wild type → Alpha →
Delta → Omicron).

In [4]:
monthly = pd.read_sql_query("""
    SELECT SUBSTR(date, 1, 7) AS month,
           SUM(new_cases) AS new_cases,
           SUM(new_deaths) AS new_deaths
    FROM daily_stats WHERE country_code != 'OWID_WRL'
    GROUP BY month ORDER BY month
""", conn)
monthly["month"] = pd.to_datetime(monthly["month"] + "-01")

px.line(monthly, x="month", y="new_cases",
        title="Global monthly new cases — the pandemic waves")

In [5]:
fig = px.scatter(monthly, x="new_cases", y="new_deaths",
                 text="month", title="Cases vs deaths by month (2020 → 2024)")
fig.update_traces(textposition="top center", textfont_size=8)
fig

The months above the trend line killed disproportionately more people per
case — early 2020 (no vaccines, no treatments) and the Delta wave (mid-2021).
Omicron (2022) produced record cases but a much flatter death count.

## 3. Country severity: it's not just about case counts

In [6]:
severity = pd.read_sql_query("""
    WITH t AS (
        SELECT d.country_code,
               MAX(d.total_cases)  AS cases,
               MAX(d.total_deaths) AS deaths
        FROM daily_stats d
        JOIN countries c USING (country_code)
        WHERE c.continent IS NOT NULL AND c.population >= 1_000_000
        GROUP BY d.country_code
    )
    SELECT c.location, c.continent, c.gdp_per_capita, c.median_age,
           c.life_expectancy, c.aged_65_older, c.diabetes_prevalence,
           c.cardiovasc_death_rate, c.hospital_beds_per_thousand,
           t.cases * 1e6 / c.population AS cases_per_m,
           t.deaths * 1e6 / c.population AS deaths_per_m,
           100.0 * t.deaths / t.cases    AS cfr_pct
    FROM t JOIN countries c USING (country_code)
    WHERE t.cases > 0
""", conn)

severity.nlargest(10, "deaths_per_m")[
    ["location", "continent", "cases_per_m", "deaths_per_m", "cfr_pct"]]

,location,continent,cases_per_m,deaths_per_m,cfr_pct
110,Peru,South America,"132,952.5","6,489.8",4.9
14,Bulgaria,Europe,"196,106.9","5,706.3",2.9
16,Bosnia and Herzegovina,Europe,"124,837.6","5,069.4",4.1
60,Hungary,Europe,"223,811.8","4,921.4",2.2
89,North Macedonia,Europe,"167,617.0","4,765.9",2.8
132,Slovenia,Europe,"639,944.6","4,756.5",0.7
58,Croatia,Europe,"326,805.5","4,652.7",1.4
49,Georgia,Asia,"497,709.2","4,580.2",0.9
33,Czechia,Europe,"453,775.8","4,146.1",0.9
84,Latvia,Europe,"528,334.8","4,039.1",0.8


In [7]:
fig = px.treemap(severity, path=[px.Constant("World"), "continent", "location"],
                 values="deaths_per_m", color="cfr_pct",
                 color_continuous_scale="Reds",
                 title="Deaths per million (area) vs CFR % (color)")
fig.update_layout(height=600)
fig

## 4. Vaccination rollout — who got there first, who got furthest?

In [8]:
first_vax = pd.read_sql_query("""
    SELECT c.location, c.continent, MIN(d.date) AS first_dose
    FROM daily_stats d JOIN countries c USING (country_code)
    WHERE d.new_vaccinations > 0 AND c.population >= 1_000_000
    GROUP BY d.country_code
""", conn)
first_vax["first_dose"] = pd.to_datetime(first_vax["first_dose"])
first_vax.nsmallest(10, "first_dose")[["location", "first_dose"]]

,location,first_dose
90,Norway,2020-12-09
132,United States,2020-12-14
16,Canada,2020-12-15
75,Latvia,2020-12-16
58,Israel,2020-12-20
17,Switzerland,2020-12-22
10,Bahrain,2020-12-24
18,Chile,2020-12-25
26,Czechia,2020-12-28
27,Germany,2020-12-28


In [9]:
vax_speed = pd.read_sql_query("""
    SELECT c.location, c.continent,
           MAX(d.people_fully_vaccinated_per_hundred) AS fully_vax_pct
    FROM daily_stats d JOIN countries c USING (country_code)
    WHERE c.continent IS NOT NULL AND c.population >= 1_000_000
    GROUP BY d.country_code HAVING fully_vax_pct <= 100
""", conn)

px.strip(vax_speed, x="fully_vax_pct", y="continent", color="continent",
         title="Fully vaccinated % by continent — final coverage")

## 5. India deep-dive (waves & vaccination interplay)

In [10]:
india = pd.read_sql_query("""
    SELECT date, new_cases_smoothed, new_deaths_smoothed,
           people_fully_vaccinated_per_hundred
    FROM daily_stats WHERE country_code = 'IND'
    ORDER BY date
""", conn)
for c in ("date",):
    india[c] = pd.to_datetime(india[c])

fig = px.line(india, x="date", y="new_cases_smoothed",
              title="India — daily new cases (7-day avg)")
fig.add_scatter(x=india["date"], y=india["new_deaths_smoothed"] * 50,
                mode="lines", name="new deaths ×50 (2nd axis)",
                yaxis="y2")
fig.update_layout(yaxis2={"overlaying": "y", "side": "right"})
fig

In [11]:
# did vaccination bend the curve? overlay coverage on the death curve
fig = px.area(india, x="date", y="people_fully_vaccinated_per_hundred",
              title="India — fully vaccinated % of population over time")
fig

## 6. Correlation: development indicators vs outcomes

In [12]:
corr_cols = ["gdp_per_capita", "median_age", "life_expectancy",
             "aged_65_older", "diabetes_prevalence", "cardiovasc_death_rate",
             "hospital_beds_per_thousand", "deaths_per_m", "cfr_pct"]
corr = severity[corr_cols].corr().round(2)

fig = px.imshow(corr, text_auto=True, aspect="auto",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Correlation matrix — country traits vs COVID outcomes")
fig.update_layout(height=560)
fig

In [13]:
conn.close()
print("EDA complete.")

EDA complete.
